In [ ]:
from abc import ABC, abstractmethod
import pyspark.sql.functions as F

In [ ]:
class TableWithDate(ABC):

    @property
    @abstractmethod
    def DATE_COLUMN(self) -> str:
        pass

    def available_year_months(self):
        new_df = self.df.select(F.col("year"), F.col("month")).distinct()
        return type(self)(new_df)

    def enrich_by_year_month(self):
        new_df = self.df.withColumn("year", F.year(self.DATE_COLUMN))\
                        .withColumn("month", F.month(self.DATE_COLUMN))
        return type(self)(new_df)

    def filter_by_year_months(self, year_months_df):
        new_df = self.df.join(year_months_df, on=["year", "month"], how="inner")
        return type(self)(new_df)

    def loaded_months(self):
        return self.enrich_by_year_month().available_year_months()
    
    def join_on_year_month(self, left, how):
        new_df = self.df.join(left, on=["year", "month"], how=how)
        return type(self)(new_df)

    def drop_year_month(self):
        new_df = self.df.drop("year", "month")
        return type(self)(new_df)